# SMARD: plataforma de información de la agencia fedeal de redes sobre el mercado eléctrico aleman. 
Los datos se solicitan mediante una petición GET siumple, los resultados
se filtran mediante parámetros de ruta (query params)
filtros
Parámetro:resolución
- 'hour': cada hora
- 'quarterhour': cada 15 min
- 'day': cada día
- 'week': cada semana
- 'month': cada mes
- 'year': cada año 

Posibles códigos para filtrar (valor igual para filtercopy):

- '1223': generación de electricidad: lignito
- '1224': generación de electricidad: energía nuclear
- '1225': generación de electricidad: eólica marina
- '1226': generación de electricidad: hidroeléctrica
- '1227': generación de electricidad: otra convencional
- '1228': generación de electricidad: otras energías renovables
- '4066': generación de electricidad: biomasa
- '4067': generación de electricidad: eólica terrestre
- '4068': generación de electricidad: fotovoltaica
- '4069': generación de electricidad: hulla
- '4070': generación de electricidad: almacenamiento por bombeo
- '4071': generación de electricidad: gas natural 
- '410': consumo de energía total (carga de la red) 
- '4359': consumo de energía total: carga residual
- '4387': consumo de energía: almacenamiento bombeado

 Parámetro: región (el parámetro regioCopy debe coincidir con el de region)
 País / zona regulatoria / área de mercado
 - DE - Alemania
 - AT - Austria
 - LU - Luxemburgo 
 - DE-LU - Zona de mercado: DE/LU (desde el 01/10/2018) 
 - DE-AT-LU - Zona de mercado DE/AT/LU (Hasta el 30/09/2018)
 - 50Hertz - zona de control (DE): 50Hertz
 - Zona de control 'Amprion' (DE): Amprion
 - 'TenneT' - Zona de control (DE): TenneT
 - 'TransnetBW' - Zona de control (DE): TransnetBW
 - 'APG' - Zona de control (AT): APG
 - 'Creos' - Zona de control (LU): Creos

In [1]:
import sys
print(sys.executable)

c:\Users\JUAN DAVID\Desktop\Proyecto Integrador FULL\Repaso desarrollo web\energy_forecasting_platform\.venv\Scripts\python.exe


In [3]:
import requests # para manejar las solicitudes
import pandas as pd # para crear datasets
from pydantic import (
    BaseModel, # para instanciar el validador
    field_validator, # validador personalizado de campo
    ValidationError # excepión de validación
)
from datetime import datetime, date
import io
import uuid
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import json

Se define el validador para verificar que los datos obtenidos por la API si sean una lista de enteros (esto para el caso de la primera llamada)

In [4]:
# se evalúa el tipado (el campo se llama igual que como viene de 
# la respuesta de la API)
# validador para los datos de la petición de la url 1
class SMARDIndex_timestamps(BaseModel):
    timestamps: list[int] # lista de enteros, ms desde 1970

    @field_validator("timestamps") #validador del campo timestamps
    @classmethod
    def validate_timestamps(cls, values: list[int]) -> list[int]:

        # si no hay valores, devuelve error de valores
        if not values:
            raise ValueError("La llamada devolvió una lista vacía")

        # valida si algun índice de los devueltos es <= 0
        if any(v <= 0 for v in values):
            raise ValueError("Existen timestamps negativos")

        # valida si la lista de timestamps no está ordenada
        if values != sorted(values):
            raise ValueError("los timestamps no vienen ordenados por fecha")  

        return values


se disponibilizan los códigos en un diccionario para llamadas automatizadas

In [5]:
# conjunto para las temporalidades disponibles
resolution_param : set = [
    'hour','quarterhour','day','week','month','year'
]

# posibles códigos para filtrar
filter_param : set = [
    '1223','1224','1225','1226','1227','1228','4066','4067','4068',
    '4069','4070','4071','410','4359','4387'
]

# parámetro de regiones disponibles
region_param: set = [
    'DE','AT','LU','DE-LU','DE-AT-LU','50Hertz'
]

# conjuntos de parametros de testeo
resolution_param_test : list = ['day']
filter_param_test: set = ['410']
region_param_test: set = ['DE']

Ahora se arma la url con la que se hará la petición

In [6]:
# ahora se arma la url para hacer la solicitud
url_time_indexes : str = f"https://smard.de/app/chart_data/{filter_param_test[0]}/{region_param_test[0]}/index_{resolution_param_test[0]}.json"
print(url_time_indexes)

https://smard.de/app/chart_data/410/DE/index_day.json


Ahora se hace la solicitud

In [7]:
# solicitud de la url
# se inicializa una lista vacía para almacenar las fechas 
# disponibles para solicitud (por año)

indexes: None
try:
    response_indexes = requests.get(url_time_indexes, timeout = 20)
    indexes = dict(response_indexes.json())
    indexes_converted = [datetime.fromtimestamp(int(el)/1000) for el in indexes["timestamps"]]
    print("índices en unix: ", indexes)

# error de tiempo de respuesta: la url se demora mas de 
# 20s en responder
except requests.exceptions.Timeout as e:
    raise TimeoutError(f'La url {url_time_indexes} tardó demasiado en responder')

# error de conexión:
except requests.exceptions.ConnectionError as e:
    raise ConnectionError(f'No se pudo conectar') from e

# error en la solicitud HTTP
except requests.exceptions.HTTPError as e:
    raise e

# error en la solicitud
except requests.exceptions.RequestException as e:
    raise e

índices en unix:  {'timestamps': [1420066800000, 1451602800000, 1483225200000, 1514761200000, 1546297200000, 1577833200000, 1609455600000, 1640991600000, 1672527600000, 1704063600000, 1735686000000, 1767222000000]}


Ahora se guardan los elementos en un json

In [8]:
indexes

{'timestamps': [1420066800000,
  1451602800000,
  1483225200000,
  1514761200000,
  1546297200000,
  1577833200000,
  1609455600000,
  1640991600000,
  1672527600000,
  1704063600000,
  1735686000000,
  1767222000000]}

In [10]:
# se guardan los indices unix obtenidos en un json 
#with open('../test_json/timestamps_by_year_unix.json', 'w') as fp:
#    json.dump(indexes, fp)

In [11]:
# se aplica el validador de la clase a los timestamps (esto para crear un archivo en local)
#try:
#    indexes_list = SMARDIndex_timestamps.model_validate(indexes)
#    print(f"{len(indexes_list.timestamps)} timestamps válidos")
#except ValidationError as e:
#    raise e

Ahora, una vez almacenado el archivo, se validan las entradas asociadas a este con la clase creada antes 

In [ ]:
load_dotenv() # carga databricks_host y databricks_token del .env()
w = WorkspaceClient() # crea el cliente, toma el host y el token de las varibales cargadas

volumen = "/Volumes/data_api_smard/raw/smard" # ruta base del volume de raw del UC

# función para guardar el contenido 
def save_raw(content: bytes, folder: str, id_load: str) -> str:
    "guarda los bytes originales de una respuesta en raw y devuelve la ruta"
    date_today = date.today().isoformat() # fecha de hoy como texto
    directory = f"{volumen}/{folder}/fecha_ingesta={date_today}" # carpeta de destino, separada por / y por fecha
    w.files.create_directory(directory)
    route = f"{directory}/{id_load}.json"
    w.files.upload(route, io.BytesIO(content), overwrite=False)
    return route

id_load = str(uuid.uuid4()) # id único para la ejecución 
route = save_raw(indexes, "indices", id_load)
print(f"saved in {route}")

for file in w.files.list_directory_contents(f"{volumen}/indices"):
    print(file.path)